# Module 05 — Lesson 05
# Sorting Data

---

> Lesson 04 got you to the exact *rows* you cared about. But "here are the dinner bills" is still a jumbled pile until you put them in an order that answers a question -- biggest bill first, or grouped by day and then by size within each day. That ordering is `.sort_values()`, and it's how you turn a filtered table into a ranked answer.

We'll keep using `tips.csv` -- copied into this lesson's own `data/` folder, same as Lesson 04.

---
## 🎯 Learning Objectives

By the end of this lesson, you will be able to:

- Sort a DataFrame by one column, ascending or descending, with `.sort_values()`
- Sort by multiple columns at once, with a different direction per column
- Sort by the row index with `.sort_index()`
- Grab the top/bottom N rows directly with `.nlargest()` / `.nsmallest()`
- Avoid the two most common sorting mistakes: forgetting sorting isn't in-place, and mismatched `ascending` lists

In [ ]:
import pandas as pd

tips = pd.read_csv("data/tips.csv")
print(f"tips.shape: {tips.shape}")
print(tips.head())

---
## 1. Sorting by One Column

`.sort_values(by='column')` reorders every row by that column's value, smallest to largest by default. Pass `ascending=False` to flip it to largest first.

In [ ]:
by_bill_asc = tips.sort_values(by="total_bill")
print("Smallest bills first:")
print(by_bill_asc.head(3))
print()

by_bill_desc = tips.sort_values(by="total_bill", ascending=False)
print("Largest bills first:")
print(by_bill_desc.head(3))

Notice the row index on the left stays with its original row -- sorting reorders rows, it doesn't renumber them. If you want a fresh 0, 1, 2... index afterward, chain `.reset_index(drop=True)`.

In [ ]:
by_bill_desc_reindexed = tips.sort_values(by="total_bill", ascending=False).reset_index(drop=True)
print("Same sort, with a clean new index:")
print(by_bill_desc_reindexed.head(3))

---
## 2. Sorting by Multiple Columns

Pass a **list** to `by` to sort by more than one column -- pandas sorts by the first column, then uses the next column(s) to break ties. `ascending` can be a single value (applies to all) or a list matching `by`, one direction per column.

In [ ]:
# Sort by 'day' first, then by 'total_bill' within each day -- both ascending
by_day_then_bill = tips.sort_values(by=["day", "total_bill"])
print(by_day_then_bill.head(6))

In [ ]:
# Different direction per column: day A-Z, but total_bill biggest-first within each day
mixed_directions = tips.sort_values(by=["day", "total_bill"], ascending=[True, False])
print(mixed_directions.head(6))

---
## 3. Sorting by the Index: `.sort_index()`

After filtering or sorting by value, the row index often ends up shuffled. `.sort_index()` puts rows back in index order -- useful when you want to restore the original row order.

In [ ]:
shuffled = tips.sort_values(by="tip", ascending=False)
print("After sorting by tip, the index is shuffled:")
print(shuffled.head(3).index.tolist())
print()

restored = shuffled.sort_index()
print("sort_index() restores original row order:")
print(restored.head(3).index.tolist())

---
## 4. `.nlargest()` / `.nsmallest()` — Top/Bottom N, Directly

When all you want is "the top 5 rows by some column," `.nlargest(n, 'column')` is a clearer (and faster) shortcut than `.sort_values(...).head(n)`.

In [ ]:
top_5_tips = tips.nlargest(5, "tip")
print("Top 5 tips:")
print(top_5_tips)
print()

bottom_3_bills = tips.nsmallest(3, "total_bill")
print("3 smallest bills:")
print(bottom_3_bills)

---
## ⚠️ Common Mistakes

In [ ]:
# -- Mistake 1: Forgetting sort_values() doesn't sort in place ------------------

print("Mistake 1 — calling sort_values() without using its return value does nothing:")
tips_copy = tips.copy()
tips_copy.sort_values(by="total_bill")   # return value thrown away!
print("  tips_copy is UNCHANGED -- still in original order:")
print(tips_copy.head(3).index.tolist())
print()
print("  Correct: reassign the result, or pass inplace=True")
tips_copy = tips_copy.sort_values(by="total_bill")
print("  Now it's actually sorted:")
print(tips_copy.head(3).index.tolist())

In [ ]:
# -- Mistake 2: ascending list length doesn't match by list length --------------

print("Mistake 2 — ascending must have ONE entry per column in `by`:")
try:
    tips.sort_values(by=["day", "total_bill"], ascending=[True])   # only 1 value for 2 columns
except ValueError as e:
    print(f"  Caught error: {e}")
print()
print("  Correct: one True/False per column being sorted:")
print(tips.sort_values(by=["day", "total_bill"], ascending=[True, False]).head(3))

In [ ]:
# -- Mistake 3: sorting a numeric-looking column that's actually text -----------

print("Mistake 3 — text columns sort ALPHABETICALLY, not numerically:")
messy = pd.DataFrame({"code": ["9", "10", "2", "21"]})
print("  Sorting as text gives '10' before '2' before '21' before '9':")
print(messy.sort_values(by="code")["code"].tolist())
print()
print("  Correct: convert to a numeric dtype FIRST, then sort:")
messy["code"] = messy["code"].astype(int)
print(messy.sort_values(by="code")["code"].tolist())

---
## ✏️ Practice Exercises

### Exercise 1 — Starter: Single-Column Sort

Sort `tips` by `"total_bill"` in descending order. Print the first 5 rows.

In [ ]:
# Your code here

### Exercise 2 — Multi-Column Sort

Sort `tips` by `"day"` (ascending) and then by `"size"` (descending) within each day. Print the first 8 rows.

In [ ]:
# Your code here

### Exercise 3 — `.nlargest()`

Use `.nlargest()` to find the 5 rows with the highest `"tip"`. Print the result.

In [ ]:
# Your code here

### Exercise 4 — Sort and Reindex

Sort `tips` by `"tip"` descending, then reset the index so it's a clean 0, 1, 2... again. Print the first 5 rows and confirm the index starts at 0.

In [ ]:
# Your code here

### Exercise 5 — (Challenge) Three-Column Sort with Mixed Directions

Sort `tips` by `"time"` (ascending), then `"day"` (ascending), then `"total_bill"` (descending) -- all in one `.sort_values()` call. Print the first 10 rows.

In [ ]:
# Your code here

---
## 📌 Key Takeaways

- **`.sort_values(by=...)` orders rows by one or more columns** — pass a list to `by` for multi-column sorts, and a matching list to `ascending` when different columns need different directions.

- **Sorting returns a new DataFrame by default** — `df.sort_values(...)` alone changes nothing; reassign the result (`df = df.sort_values(...)`) or pass `inplace=True`.

- **`.nlargest()` / `.nsmallest()` are the direct route to "top/bottom N rows"** — clearer and more efficient than `.sort_values().head(n)` when that's all you need.

---
## What's Next?

**Lesson 06 — Handling Missing Values** tackles the gaps every real dataset has — `tips.csv` was clean, but most files you'll load from here on won't be.